# Cobertura de Mamografia (50–69 anos) — PNS 2019

Análise da variação da cobertura de mamografia entre UFs e subgrupos de renda, escolaridade e cor/raça, com comparação da evolução anual de exames SUS (2020–2024).

**Fontes:**
- `subgrupos_pns_2019.csv` — PNS 2019, subgrupos por escolaridade, cor/raça e renda
- `exames_sus_2020_2024.csv` — exames de rastreamento SUS por UF e ano
- `base_uf.csv` — indicadores consolidados por UF (incidência, mortalidade, infraestrutura)

## 1. Importação e carregamento dos dados

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pns = pd.read_csv("../dados/tratados/subgrupos_pns_2019.csv")
exames = pd.read_csv("../dados/tratados/exames_sus_2020_2024.csv")
uf = pd.read_csv("../dados/tratados/base_uf.csv")

pns_mamo = pns[pns["indicador"] == "fez_ultimos_2_anos"].copy()
pns_mamo.head()

: 

## 2. Cobertura por UF (PNS 2019)

In [ ]:
uf_sorted = uf[["uf_sigla", "uf_nome", "regiao", "cobertura_pns_2019",
                "cobertura_pns_2013", "var_cobertura_2013_2019"]] \
                .sort_values("cobertura_pns_2019", ascending=False)

uf_sorted

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(uf_sorted["uf_sigla"], uf_sorted["cobertura_pns_2019"], color="steelblue")
plt.xlabel("Cobertura de mamografia (%) - PNS 2019")
plt.title("Cobertura de mamografia por UF (PNS 2019)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

**Observação:** UFs do Sudeste/Sul (ex. SP 69,3%, DF 61,8%, PR 58,3%) apresentam cobertura mais alta; UFs do Norte (ex. AP 34,9%, RR 37,4%, AC 40,2%) apresentam cobertura mais baixa, refletindo desigualdade regional.

## 3. Cobertura por escolaridade

In [ ]:
esc = pns_mamo[pns_mamo["dimensao"] == "escolaridade"]

ordem_esc = [
    "Sem instrução e fundamental incompleto",
    "Fundamental completo e médio incompleto",
    "Médio completo e superior incompleto",
    "Superior completo",
]

pivot_esc = esc.pivot_table(index="categoria", columns="regiao", values="percentual")
pivot_esc = pivot_esc.reindex(ordem_esc)
pivot_esc

In [ ]:
pivot_esc.plot(kind="bar", figsize=(10, 6))
plt.ylabel("% fez mamografia (últimos 2 anos)")
plt.title("Cobertura de mamografia por escolaridade e região")
plt.tight_layout()
plt.show()

**Padrão:** cobertura cresce de forma quase monotônica com a escolaridade — no Brasil, de 49,0% (sem instrução/fundamental incompleto) para 77,8% (superior completo), gradiente de ~29 p.p.

## 4. Cobertura por cor/raça

In [ ]:
raca = pns_mamo[pns_mamo["dimensao"] == "cor_raca"]
pivot_raca = raca.pivot_table(index="categoria", columns="regiao", values="percentual")
pivot_raca

In [ ]:
pivot_raca.plot(kind="bar", figsize=(10, 6))
plt.ylabel("% fez mamografia (últimos 2 anos)")
plt.title("Cobertura de mamografia por cor/raça e região")
plt.tight_layout()
plt.show()

**Padrão:** no Brasil, mulheres brancas têm cobertura maior (61,8%) do que pardas (54,4%) e pretas (56,5%). A diferença é mais acentuada em regiões como Sudeste e Sul.

## 5. Cobertura por renda

In [ ]:
renda = pns_mamo[pns_mamo["dimensao"] == "rendimento"]

ordem_renda = [
    "Sem rendimento até 1/4 SM", "De 1/4 a 1/2 SM", "De 1/2 a 1 SM",
    "De 1 a 2 SM", "De 2 a 3 SM", "De 3 a 5 SM", "Mais de 5 SM",
]

pivot_renda = renda.pivot_table(index="categoria", columns="regiao", values="percentual")
pivot_renda = pivot_renda.reindex(ordem_renda)
pivot_renda

In [ ]:
pivot_renda.plot(kind="line", marker="o", figsize=(10, 6))
plt.ylabel("% fez mamografia (últimos 2 anos)")
plt.title("Cobertura de mamografia por faixa de renda e região")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

**Padrão:** o gradiente de renda é o mais forte dos três — no Brasil, varia de 42,9% (sem rendimento a 1/4 SM) até 83,7% (mais de 5 SM), diferença de ~41 p.p.

## 6. Relação entre cobertura (PNS) e oferta de exames/equipamentos (SUS)

In [ ]:
infra = uf[["uf_sigla", "cobertura_pns_2019", "mamografos_por_100mil_mulheres",
            "exames_sus_2024", "pct_tratados_ate_60_dias"]]

corr = infra[["cobertura_pns_2019", "mamografos_por_100mil_mulheres",
              "pct_tratados_ate_60_dias"]].corr()
corr

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(infra["mamografos_por_100mil_mulheres"], infra["cobertura_pns_2019"])
for _, row in infra.iterrows():
    plt.annotate(row["uf_sigla"],
                 (row["mamografos_por_100mil_mulheres"], row["cobertura_pns_2019"]))
plt.xlabel("Mamógrafos por 100 mil mulheres (2024)")
plt.ylabel("Cobertura PNS 2019 (%)")
plt.title("Infraestrutura vs. cobertura autorreferida")
plt.tight_layout()
plt.show()

## 7. Evolução anual de exames SUS por região (2020–2024)

In [ ]:
exames_regiao_ano = exames.groupby(["regiao", "ano"])["n_mamografias_rastreamento_50_69"] \
    .sum().reset_index()

pivot_ano = exames_regiao_ano.pivot(index="ano", columns="regiao",
                                     values="n_mamografias_rastreamento_50_69")
pivot_ano

In [ ]:
pivot_ano.plot(kind="line", marker="o", figsize=(10, 6))
plt.ylabel("Nº de exames de mamografia (rastreamento, 50-69 anos)")
plt.title("Evolução anual de exames SUS por região (2020–2024)")
plt.tight_layout()
plt.show()

## 8. Crescimento acumulado 2020 → 2024 por região

In [ ]:
crescimento = ((pivot_ano.loc[2024] - pivot_ano.loc[2020]) / pivot_ano.loc[2020] * 100) \
    .sort_values(ascending=False)
crescimento

In [ ]:
plt.figure(figsize=(9, 6))
crescimento.plot(kind="bar", color="darkorange")
plt.ylabel("Crescimento acumulado 2020→2024 (%)")
plt.title("Crescimento acumulado de exames SUS por região")
plt.axhline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()

**Comparação entre anos e regiões:**
- **Centro-Oeste** e **Nordeste** têm o maior crescimento acumulado (acima de 100%), partindo de uma base de exames menor em 2020.
- **Sudeste** e **Sul** crescem de forma mais moderada e, em alguns casos (ex. Sul, Minas Gerais), apresentam **queda entre 2023 e 2024** — possível reflexo de normalização pós-pandemia ou ajuste de demanda reprimida.
- O crescimento mais acelerado em regiões historicamente com menor cobertura (Norte/Nordeste) indica uma trajetória de redução de desigualdade regional no acesso a exames pelo SUS, embora os níveis absolutos ainda sejam muito inferiores aos do Sudeste.

## 9. Síntese dos achados

| Dimensão | Gradiente observado (Brasil) | Amplitude |
|---|---|---|
| Escolaridade | 49,0% (sem instrução) → 77,8% (superior) | ~29 p.p. |
| Cor/raça | 54,4% (parda) → 61,8% (branca) | ~7 p.p. |
| Renda | 42,9% (sem rend. a ¼ SM) → 83,7% (>5 SM) | ~41 p.p. |
| UF (Norte vs. Sudeste) | ~35–40% (Norte) → ~65–70% (Sudeste/DF) | ~30 p.p. |

**Conclusão:** a renda é o fator de maior desigualdade na cobertura de mamografia, seguido pela escolaridade e pela desigualdade regional (Norte/Nordeste vs. Sul/Sudeste). A diferença por cor/raça é mais modesta, mas consistente — mulheres pretas e pardas sempre abaixo das brancas na maioria das regiões. A série 2020–2024 de exames SUS mostra convergência parcial, com Nordeste e Centro-Oeste crescendo mais rápido, embora ainda distantes dos volumes absolutos do Sudeste.